In [ ]:
# ================== 0) Setup & paths ==================
import os, gc, json, random
import numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests

sns.set_context("talk"); sns.set_style("whitegrid")

DATA_DIR    = "./output_embeddings"
FEATURE_DIR = "./tda_feature_outputs"
BASE_FEAT   = os.path.join(FEATURE_DIR, "features_base.parquet")
OUTPUT_DIR  = "./tda_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIME_FEATURES_PATH = None

SEED   = 42
N_PERM = 5000
ALPHA  = 0.05
rng    = np.random.default_rng(SEED)

TDA_FEATURES = [
    "betti_L1","betti_L2",
    "H1_sum_pers","H1_max_pers","H1_n_bars","H0_sum_pers","H0_n_bars",
    "ratio_sum_H1_H0","ratio_count_H1_H0","midlife_gap_H1_H0","entropy_gap_H1_H0",
    "BettiH1_AUC","BettiH1_peak_loc_norm",
    "PI_H1_long","PI_H1_short","PI_H1_long_short_ratio",
]

print("[cfg] BASE_FEAT:", BASE_FEAT)

In [ ]:
# 1) Index computation
SLEEP_STAGES_ASLEEP = {1, 2, 3, 4}  # 0=Wake, others = asleep

def session_paths(sid: str, data_dir: str = DATA_DIR) -> dict:
    return {
        "evt1": os.path.join(data_dir, f"{sid}_event1.npy"),
        "evt2": os.path.join(data_dir, f"{sid}_event2.npy"),
        "sleep": os.path.join(data_dir, f"{sid}_sleep.npy"),
    }

def compute_index_from_epoch_labels(sid: str) -> dict:
    p = session_paths(sid)
    if not all(os.path.exists(v) for v in p.values()):
        return {"id": sid, "index_all": np.nan, "TST_hours": np.nan, "n_events": np.nan}

    evt1 = np.load(p["evt1"])
    evt2 = np.load(p["evt2"])
    sleep = np.load(p["sleep"])

    asleep_epochs = np.isin(sleep.astype(int), list(SLEEP_STAGES_ASLEEP)).sum()
    TST_hours = asleep_epochs / 120.0
    if TST_hours <= 0:
        return {"id": sid, "index_all": np.nan, "TST_hours": 0.0, "n_events": 0}

    n_events = np.logical_or(evt1 > 0, evt2 > 0).sum()
    index_all = n_events / TST_hours

    return {"id": sid, "index_all": float(index_all), "TST_hours": float(TST_hours), "n_events": int(n_events)}

def group_index(val: float) -> tuple[int, str]:
    """Return (group_id, group_name) per thresholds."""
    if not np.isfinite(val):
        return (-1, "missing")
    if val < 1:   return (0, "grp0(<1)")
    if val < 5:   return (1, "grp1(1-5)")
    if val < 10:  return (2, "grp2(5-10)")
    else:         return (3, "grp3(>=10)")

df_feat = pd.read_parquet(BASE_FEAT)
ids = df_feat["id"].astype(str).unique().tolist()
rows = [compute_index_from_epoch_labels(sid) for sid in ids]
df_idx = pd.DataFrame(rows)

grp_id, grp_nm = [], []
for v in df_idx["index_all"].values:
    g, n = group_index(v)
    grp_id.append(g); grp_nm.append(n)
df_idx["grp_id"] = grp_id
df_idx["grp_name"] = grp_nm

print(df_idx.describe(include="all"))
print(df_idx["grp_name"].value_counts().to_string())

map_path = os.path.join(OUTPUT_DIR, "index_mapping.csv")
df_idx.to_csv(map_path, index=False)
print("[save] index map ->", map_path)

In [ ]:
# 2) Merge with feature tables
df_base = df_feat.drop(columns=[c for c in ["y_sess","label_name","source"] if c in df_feat.columns], errors="ignore")

if TIME_FEATURES_PATH and os.path.exists(TIME_FEATURES_PATH):
    df_time = pd.read_parquet(TIME_FEATURES_PATH) if TIME_FEATURES_PATH.endswith(".parquet") else pd.read_csv(TIME_FEATURES_PATH)
    df_time = df_time.add_prefix("time_").rename(columns={"time_id": "id"})
    df_merged = df_base.merge(df_time, on="id", how="left")
else:
    df_merged = df_base.copy()

df_merged = df_merged.merge(
    df_idx[["id", "index_all", "grp_id", "grp_name", "TST_hours", "n_events"]],
    on="id", how="left"
)

df_merged = df_merged[df_merged["grp_id"].between(0, 3)].reset_index(drop=True)
print("[info] merged shape:", df_merged.shape)

In [ ]:
# ================== (NEW) Build GLOBAL time features ==================
TIME_KEEP_PER_DIM = False
TIME_MAX_DIMS     = 16

def time_feature_path(sid: str, data_dir: str = DATA_DIR) -> str:
    return os.path.join(data_dir, f"{sid}_time_features.npy")

def summarize_time_array(A: np.ndarray) -> dict:
    A = np.asarray(A, dtype=float)
    out = {}
    if A.ndim == 1:
        v = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)
        out.update({
            "time_f1": float(np.mean(v)),
            "time_f2": float(np.std(v, ddof=1)),
            "time_f3": float(np.linalg.norm(v)),
        })
    elif A.ndim == 2:
        A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)
        mu = np.mean(A, axis=0)
        sd = np.std(A, axis=0, ddof=1)
        out.update({
            "time_f1": float(np.mean(mu)),
            "time_f2": float(np.std(mu, ddof=1)),
            "time_f3": float(np.mean(sd)),
            "time_f4": float(np.std(sd, ddof=1)),
            "time_f5": float(np.linalg.norm(mu)),
            "time_f6": float(np.linalg.norm(sd)),
        })
    else:
        v = np.nan_to_num(A.reshape(-1), nan=0.0, posinf=0.0, neginf=0.0)
        out.update({
            "time_f1": float(np.mean(v)),
            "time_f2": float(np.std(v, ddof=1)),
            "time_f3": float(np.linalg.norm(v)),
        })
    return out

def build_time_df_from_files(ids) -> pd.DataFrame:
    rows = []
    for sid in ids:
        p = time_feature_path(sid)
        if not os.path.exists(p):
            continue
        try:
            arr = np.load(p, allow_pickle=False)
            feats = summarize_time_array(arr)
            feats["id"] = str(sid)
            rows.append(feats)
        except Exception as e:
            print(f"[time warn] {sid}: {e}")
    if not rows:
        return pd.DataFrame(columns=["id"])
    return pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Build table for IDs present in base feature file
ids = df_feat["id"].astype(str).unique().tolist()
df_time = build_time_df_from_files(ids)
print("[info] time feature shape:", df_time.shape)

# Merge with base + index map
df_base = df_feat.drop(columns=[c for c in ["y_sess","label_name","source"] if c in df_feat.columns], errors="ignore")
df_merged = (
    df_base.merge(df_time, on="id", how="left")
           .merge(df_idx[["id","index_all","grp_id","grp_name","TST_hours","n_events"]],
                  on="id", how="left")
)
df_merged = df_merged[df_merged["grp_id"].between(0,3)].reset_index(drop=True)
print("[info] merged shape (base + time):", df_merged.shape)

# Fixed feature lists (6 + 6)
TDA_FEATURES = ["tda_f1","tda_f2","tda_f3","tda_f4","tda_f5","tda_f6"]
TIME_FEATURES = ["time_f1","time_f2","time_f3","time_f4","time_f5","time_f6"]
ALL_FEATURES = [f for f in TDA_FEATURES + TIME_FEATURES if f in df_merged.columns]
print(f"[info] testing {len(ALL_FEATURES)} features")

In [ ]:
df_merged

In [ ]:
# ================== (UPDATED) Permutation tests ==================
feat_cols = ALL_FEATURES
results = []
g_all = df_merged["grp_id"].astype(int).values

for c in feat_cols:
    v = df_merged[c].astype(float).values
    m = np.isfinite(v)
    x = v[m]; g = g_all[m]
    if len(x) < 10:
        continue

    # Global KW across groups
    H = stat_kw_H(x, g)
    p_kw, null_kw = perm_p_value(H, stat_kw_H, x, g, N_PERM, rng)
    results.append(dict(feature=c, test="PermKW_global", stat=H, p=p_kw, q=np.nan,
                        effect_median_diff=np.nan, cliffs_delta=np.nan, n_perm=len(null_kw)))

    # One-vs-rest MWU (per group)
    for k in sorted(np.unique(g)):
        obs = stat_mwu_umax_ova(x, g, pos_label=k)
        p_mwu, null_mwu = perm_p_value(
            obs, lambda xv, gv: stat_mwu_umax_ova(xv, gv, pos_label=k), x, g, N_PERM, rng
        )
        xk, xr = x[g==k], x[g!=k]
        eff_md = float(np.median(xk) - np.median(xr)) if len(xk) and len(xr) else np.nan
        try: 
            delta = cliffs_delta(xk, xr)
        except Exception: 
            delta = np.nan
        results.append(dict(feature=c, test=f"PermMWU_group{k}_vs_all", stat=obs, p=p_mwu, q=np.nan,
                            effect_median_diff=eff_md, cliffs_delta=delta, n_perm=len(null_mwu)))

res = pd.DataFrame(results)
if res.empty:
    print("[warn] no tests ran.")
else:
    for tname, sub in res.groupby("test", sort=False):
        _, qvals, *_ = multipletests(sub["p"].values, alpha=ALPHA, method="fdr_bh")
        res.loc[sub.index, "q"] = qvals
    res.sort_values(["q","p","feature"], inplace=True)

    out_csv = os.path.join(OUTPUT_DIR, "significance_groups_TDAplusTIME.csv")
    res.to_csv(out_csv, index=False)
    print("[save] significance ->", out_csv)
    print(res[["feature","test","stat","p","q","n_perm","effect_median_diff","cliffs_delta"]]
          .head(40).to_string(index=False))

In [ ]:
# --- EHR-only permutation tests ---

feat_cols = EHR_FEATURES
g_all = df_merged["grp_id"].astype(int).values

results = []
for c in feat_cols:
    v = df_merged[c].astype(float).values
    m = np.isfinite(v)
    x = v[m]; g = g_all[m]
    if len(x) < 10:
        continue

    # Global KW across groups
    H = stat_kw_H(x, g)
    p_kw, null_kw = perm_p_value(H, stat_kw_H, x, g, N_PERM, rng)
    results.append(dict(
        feature=c, test="PermKW_global", stat=H, p=p_kw, q=np.nan,
        effect_median_diff=np.nan, cliffs_delta=np.nan, n_perm=len(null_kw)
    ))

    # One-vs-rest MWU
    for k in sorted(np.unique(g)):
        obs = stat_mwu_umax_ova(x, g, pos_label=k)
        p_mwu, null_mwu = perm_p_value(
            obs, lambda xv, gv: stat_mwu_umax_ova(xv, gv, pos_label=k),
            x, g, N_PERM, rng
        )
        xk, xr = x[g==k], x[g!=k]
        eff_md = float(np.median(xk) - np.median(xr)) if len(xk) and len(xr) else np.nan
        try:
            delta = cliffs_delta(xk, xr)
        except Exception:
            delta = np.nan
        results.append(dict(
            feature=c, test=f"PermMWU_group{k}_vs_all", stat=obs, p=p_mwu, q=np.nan,
            effect_median_diff=eff_md, cliffs_delta=delta, n_perm=len(null_mwu)
        ))

res_ehr = pd.DataFrame(results)

if res_ehr.empty:
    print("[warn] no tests ran.")
else:
    for tname, sub in res_ehr.groupby("test", sort=False):
        _, qvals, *_ = multipletests(sub["p"].values, alpha=0.05, method="fdr_bh")
        res_ehr.loc[sub.index, "q"] = qvals

    res_ehr.sort_values(["q","p","feature"], inplace=True)

    out_csv = os.path.join(OUTPUT_DIR, "significance_groups_EHR.csv")
    res_ehr.to_csv(out_csv, index=False)
    print("[save] EHR-only significance ->", out_csv)
    display(res_ehr[["feature","test","stat","p","q","n_perm","effect_median_diff","cliffs_delta"]].head(40))

In [ ]:
## Visualizations

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11

TIME_FEATURES = ["time_mean", "time_std", "time_l2"]

def plot_time_features_by_group(df_merged):
    """Visualize time features across groups"""
    
    available_feats = [f for f in TIME_FEATURES if f in df_merged.columns]
    if not available_feats:
        print("No time features found in dataframe")
        return
        
    print(f"Plotting {len(available_feats)} time features across groups")
    
    for feat in available_feats:
        d = df_merged[["grp_name", feat]].dropna()
        if d.empty:
            continue
            
        groups = sorted(d["grp_name"].unique())
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f"Time Feature: {feat}\nDistribution across Groups", fontsize=16, fontweight='bold')
        
        # Boxplot + points
        sns.boxplot(data=d, x="grp_name", y=feat, ax=axes[0,0], showfliers=False, order=groups)
        sns.stripplot(data=d, x="grp_name", y=feat, ax=axes[0,0], alpha=0.5, size=3, order=groups)
        axes[0,0].set_title("Boxplot with Data Points")
        axes[0,0].tick_params(axis='x', rotation=45)
        for i, group in enumerate(groups):
            n = len(d[d["grp_name"] == group])
            axes[0,0].text(i, d[feat].max() * 1.02, f'n={n}', ha='center', va='bottom', fontsize=9)
        
        # ECDF
        colors = plt.cm.Set3(np.linspace(0, 1, len(groups)))
        for i, group in enumerate(groups):
            group_data = d[d["grp_name"] == group][feat].values
            sns.ecdfplot(group_data, ax=axes[0,1], label=group, color=colors[i], linewidth=2)
        axes[0,1].legend(title="Group")
        axes[0,1].set_title("Empirical Cumulative Distribution")
        axes[0,1].set_xlabel(feat)
        axes[0,1].set_ylabel("Cumulative Probability")
        
        # Density
        for i, group in enumerate(groups):
            group_data = d[d["grp_name"] == group][feat].values
            if len(group_data) > 1:
                density = gaussian_kde(group_data)
                x_vals = np.linspace(d[feat].min(), d[feat].max(), 100)
                axes[1,0].plot(x_vals, density(x_vals), label=group, color=colors[i], linewidth=2, alpha=0.8)
        axes[1,0].legend(title="Group")
        axes[1,0].set_title("Probability Density")
        axes[1,0].set_xlabel(feat)
        axes[1,0].set_ylabel("Density")
        
        # Mean ± CI
        means, cis, labels = [], [], []
        for group in groups:
            group_data = d[d["grp_name"] == group][feat].values
            if len(group_data) > 0:
                mean_val = np.mean(group_data)
                se_val = np.std(group_data) / np.sqrt(len(group_data)) if len(group_data) > 1 else 0
                ci_val = 1.96 * se_val
                means.append(mean_val); cis.append(ci_val); labels.append(group)
        
        axes[1,1].errorbar(range(len(means)), means, yerr=cis, fmt='o',
                           capsize=5, capthick=2, markersize=8, linewidth=2)
        axes[1,1].set_xticks(range(len(means)))
        axes[1,1].set_xticklabels(labels, rotation=45)
        axes[1,1].set_title("Mean ± 95% CI")
        axes[1,1].set_xlabel("Group")
        axes[1,1].set_ylabel(f"Mean {feat}")
        axes[1,1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Summary stats
        print(f"\n{feat} - Summary Statistics:")
        for group in groups:
            group_data = d[d["grp_name"] == group][feat]
            if len(group_data) > 0:
                print(f"  {group}: n={len(group_data)}, mean={group_data.mean():.3f} ± {group_data.std():.3f}")

# Run
plot_time_features_by_group(df_merged)

In [ ]:
import os

outdir = "./tda_analysis_outputs"
os.makedirs(outdir, exist_ok=True)

def plot_time_features_by_group_pretty(
    df_merged,
    features=("time_f1",),
    group_col="grp_name",
    outdir=outdir,
    zoom_quantiles=(0.02, 0.98),
    show_points=True,
    pdf_xlim=(-1, 1),
    ecdf_xlim=(-1, 1)
):
    import numpy as np, seaborn as sns, matplotlib.pyplot as plt
    from scipy.stats import gaussian_kde

    sns.set_style("whitegrid")
    palette = sns.color_palette("tab10")

    groups = sorted(df_merged[group_col].unique())
    color_map = {g: palette[i % len(palette)] for i, g in enumerate(groups)}

    for feat in [f for f in features if f in df_merged.columns]:
        d = df_merged[[group_col, feat]].dropna()
        if d.empty:
            continue

        # ---------- Box + strip ----------
        lo, hi = np.nanquantile(d[feat], zoom_quantiles)
        pad = 0.05 * (hi - lo) if hi > lo else 1.0
        fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
        sns.boxplot(data=d, x=group_col, y=feat, order=groups,
                    ax=ax, showfliers=False, width=0.55, linewidth=1.25)
        if show_points:
            sns.stripplot(data=d, x=group_col, y=feat, order=groups,
                          ax=ax, alpha=0.45, size=2.8, jitter=0.15, color="k")
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_title("Distribution by Group")
        ax.set_xlabel("Group"); ax.set_ylabel(feat)
        path = os.path.join(outdir, f"{feat}_box.pdf")
        fig.savefig(path, dpi=450); plt.close(fig)
        print(f"Saved {path}")

        # ---------- ECDF ----------
        fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
        for g in groups:
            vals = d.loc[d[group_col] == g, feat].to_numpy()
            if vals.size:
                sns.ecdfplot(vals, ax=ax, label=str(g),
                             color=color_map[g], linewidth=2)
        ax.set_xlim(*ecdf_xlim)
        ax.set_title("Empirical CDF")
        ax.set_xlabel(feat); ax.set_ylabel("Proportion")
        ax.legend(title="Group", frameon=True, framealpha=0.9,
                  facecolor="white", fontsize=9, title_fontsize=10, loc="lower right")
        path = os.path.join(outdir, f"{feat}_ecdf.pdf")
        fig.savefig(path, dpi=450); plt.close(fig)
        print(f"Saved {path}")

        # ---------- KDE ----------
        fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
        X = np.linspace(pdf_xlim[0], pdf_xlim[1], 500)
        for g in groups:
            vals = d.loc[d[group_col] == g, feat].to_numpy()
            vals = vals[np.isfinite(vals)]
            if vals.size > 1:
                kde = gaussian_kde(vals)
                ax.plot(X, kde(X), label=str(g),
                        color=color_map[g], linewidth=2)
        ax.set_xlim(*pdf_xlim)
        ax.set_title("Probability Density")
        ax.set_xlabel(feat); ax.set_ylabel("Density")
        ax.legend(title="Group", frameon=True, framealpha=0.9,
                  facecolor="white", fontsize=9, title_fontsize=10,
                  loc="upper right", ncol=min(len(groups), 2))
        path = os.path.join(outdir, f"{feat}_kde.pdf")
        fig.savefig(path, dpi=450); plt.close(fig)
        print(f"Saved {path}")

        # ---------- Mean ± CI ----------
        fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
        means, cis, labels = [], [], []
        for g in groups:
            vals = d.loc[d[group_col] == g, feat].to_numpy()
            if vals.size:
                m = float(np.nanmean(vals))
                se = float(np.nanstd(vals, ddof=1)) / np.sqrt(np.count_nonzero(np.isfinite(vals))) if vals.size > 1 else 0.0
                means.append(m); cis.append(1.96 * se); labels.append(str(g))
        ax.errorbar(range(len(means)), means, yerr=cis, fmt="o",
                    capsize=5, capthick=1.6, markersize=6.5, linewidth=2)
        ax.set_xticks(range(len(means))); ax.set_xticklabels(labels)
        ax.set_title("Mean ± 95% CI")
        ax.set_xlabel("Group"); ax.set_ylabel(feat)
        ax.grid(alpha=0.3)
        path = os.path.join(outdir, f"{feat}_meanCI.pdf")
        fig.savefig(path, dpi=450); plt.close(fig)
        print(f"Saved {path}")

In [ ]:
plot_time_features_by_ahi_pretty(
    df_merged,
    features=("time_mean",),   # you can add "time_l2" etc if you want
)